# 04 · Export to ONNX

Exports the best checkpoint (YOLO `.pt` or RF-DETR `.pth`) to ONNX and copies it to `../backend/models/leaf_disease_seg.onnx`.

In [1]:
import os
import shutil
from pathlib import Path

import onnxruntime as ort
from dotenv import load_dotenv

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")

DATA_VERSION = f"v{os.environ['ROBOFLOW_VERSION']}"
RUN_TAG = ""  # "" = runs/yolo26x-seg_data-v2 (trained Nov 4); "_lds" = a retrain with the 02 settings
CHECKPOINT = ROOT / "runs" / f"yolo26x-seg_data-{DATA_VERSION}{RUN_TAG}" / "weights" / "best.pt"
# CHECKPOINT = ROOT / "runs" / f"rfdetr-seg-medium-default_data-{DATA_VERSION}" / "checkpoint_best_total.pth"
BACKEND_MODEL = ROOT.parent / "backend" / "models" / "leaf_disease_seg.onnx"
print("checkpoint:", CHECKPOINT.relative_to(ROOT))

checkpoint: runs\yolo26x-seg_data-v2\weights\best.pt


In [2]:
if CHECKPOINT.suffix == ".pt":
    from ultralytics import YOLO

    model = YOLO(CHECKPOINT)
    imgsz = model.ckpt["train_args"]["imgsz"]  # export at the training size
    onnx_path = model.export(format="onnx", imgsz=imgsz)  # written next to the .pt
else:
    import rfdetr

    onnx_path = rfdetr.RFDETR.from_checkpoint(CHECKPOINT).export(output_dir=str(CHECKPOINT.parent / "export"))
print(onnx_path)

Ultralytics 8.4.184  Python-3.13.14 torch-2.11.0+cu128 CPU (Intel Core Ultra 9 285K)


YOLO26x-seg summary (fused): 204 layers, 62,730,197 parameters, 0 gradients, 804.9 GFLOPs



PyTorch: starting from 'C:\Users\bp67339\PCPL\plant disease detection\training\runs\yolo26x-seg_data-v2\weights\best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) ((1, 39, 21504), (1, 32, 256, 256)) (135.3 MB)



ONNX: starting export with onnx 1.23.2 opset 18...


ONNX: slimming with onnxslim 0.1.98...


ONNX: export success  8.6s, saved as 'C:\Users\bp67339\PCPL\plant disease detection\training\runs\yolo26x-seg_data-v2\weights\best.onnx' (239.9 MB)



Export complete (10.8s)
Results saved to C:\Users\bp67339\PCPL\plant disease detection\training\runs\yolo26x-seg_data-v2\weights\best.onnx
Predict:         yolo predict task=segment model=C:\Users\bp67339\PCPL\plant disease detection\training\runs\yolo26x-seg_data-v2\weights\best.onnx imgsz=1024 
Validate:        yolo val task=segment model=C:\Users\bp67339\PCPL\plant disease detection\training\runs\yolo26x-seg_data-v2\weights\best.onnx imgsz=1024 data=c:\Users\bp67339\PCPL\plant disease detection\training\datasets\lds_v1-v2-yolo\data.yaml 
Visualize:       https://netron.app


C:\Users\bp67339\PCPL\plant disease detection\training\runs\yolo26x-seg_data-v2\weights\best.onnx


In [3]:
BACKEND_MODEL.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(onnx_path, BACKEND_MODEL)
print(f"{BACKEND_MODEL}  ({BACKEND_MODEL.stat().st_size / 1e6:.1f} MB)")

C:\Users\bp67339\PCPL\plant disease detection\backend\models\leaf_disease_seg.onnx  (251.6 MB)


## Check the exported model loads

These input/output names and shapes are what `backend/app/inference.py` has to decode. They differ per model:
- YOLO-seg: `images [1, 3, S, S]` (S = training `imgsz`, 1024 by default) → `output0` (boxes + class scores + 32 mask coefficients per candidate) and
  `output1 [1, 32, S/4, S/4]` (mask prototypes); masks = sigmoid(coefficients · prototypes), cropped to the box and resized.
- RF-DETR-Seg: boxes, class logits and per-query masks directly.

In [4]:
session = ort.InferenceSession(str(BACKEND_MODEL), providers=["CPUExecutionProvider"])
for i in session.get_inputs():
    print("input ", i.name, i.shape, i.type)
for o in session.get_outputs():
    print("output", o.name, o.shape, o.type)

input  images [1, 3, 1024, 1024] tensor(float)
output output0 [1, 39, 21504] tensor(float)
output output1 [1, 32, 256, 256] tensor(float)
